# Project interactive visualization
Using a dataset that your group is consider using for the term project, let's build some meaningful user-driven data visualization. Depending on your dataset this could include:
- Usage of geospatial information
- Building interactive views with widgets
- Organize multiple components into a simple dashboard

Keep these design principles in mind:
While you navigate through this notebook some things to take into consideration:
* Do not add interaction just to add it. Make sure it helps answer a question.
* Use meaningful titles and labels
* Document your code so it's readable and clean. If something does not work, document the issue and explain your best attempt.

In [ ]:
# Install jupyter_bokeh library and upgrade plotly library.
!pip install jupyter_bokeh
!pip install plotly --upgrade

In [ ]:
## These are most likely the libraries you will use
# Add or remove imports as needed

import pandas as pd
import numpy as np

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go

# Geospatial / geocoding
from geopy.geocoders import Nominatim

# Panel
import panel as pn
pn.extension('plotly')

import itertools

In [ ]:
### Load your dataset here

# 1. Define the list of seasons we want to collect
seasons = ["2024-25", "2025-26"]

all_dfs = []

# 2. Loop through each season and read the data
for season in seasons:
    # Construct the URL dynamically
    url = f"https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/{season}/gws/merged_gw.csv"

    try:
        # Some older files might need 'latin-1' encoding
        temp_df = pd.read_csv(url, encoding='utf-8')
    except UnicodeDecodeError:
        temp_df = pd.read_csv(url, encoding='latin-1')
    except Exception as e:
        print(f"Could not download data for {season}: {e}")
        continue

    # 3. Add a season column so you can distinguish the data later
    temp_df['season'] = season

    # Append to our list
    all_dfs.append(temp_df)
    print(f"Successfully loaded {season}")

# 4. Combine all DataFrames into one
df_combined = pd.concat(all_dfs, ignore_index=True)

# View the result
print(f"\nFinal combined shape: {df_combined.shape}")
print(df_combined['season'].value_counts())
df_combined.head()

Successfully loaded 2024-25
Successfully loaded 2025-26

Final combined shape: (49958, 54)
season
2024-25    27605
2025-26    22353
Name: count, dtype: int64


,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,...,transfers_out,value,was_home,yellow_cards,GW,season,clearances_blocks_interceptions,defensive_contribution,recoveries,tackles
0,Alex Scott,MID,Bournemouth,1.6,0,0,11,0,12.8,77,...,0,50,False,0,1,2024-25,NaN,NaN,NaN,NaN
1,Carlos Miguel dos Santos Pereira,GK,Nott'm Forest,2.2,0,0,0,0,0.0,427,...,0,45,True,0,1,2024-25,NaN,NaN,NaN,NaN
2,Tomiyasu Takehiro,DEF,Arsenal,0.0,0,0,0,0,0.0,22,...,0,50,True,0,1,2024-25,NaN,NaN,NaN,NaN
3,Malcolm Ebiowei,MID,Crystal Palace,0.0,0,0,0,0,0.0,197,...,0,45,False,0,1,2024-25,NaN,NaN,NaN,NaN
4,Ben Brereton Díaz,MID,Southampton,1.0,0,0,-2,0,14.0,584,...,0,55,False,1,1,2024-25,NaN,NaN,NaN,NaN


In [ ]:
# Basic cleaning and build a player-level summary df for interactive analysis

df = df_combined.copy()

# Keep the fields needed for interactive polts
keep_cols = ["season", "name", "team", "position", "value", "total_points", "minutes", "starts"]
df = df[keep_cols].dropna(subset=["season", "name", "position", "value", "total_points"])

# Standardize text fields and keep the four main positions
df["name"] = df["name"].astype(str).str.strip()
df["team"] = df["team"].astype(str).str.strip()
df["position"] = df["position"].astype(str).str.upper().str.strip()
df = df[df["position"].isin(["GK", "DEF", "MID", "FWD"])].copy()

# Build one row per player per season
player_summary = (
    df.groupby(["season", "name", "team", "position"], as_index=False)
      .agg({
          "value": "mean",
          "total_points": "sum",
          "minutes": "sum",
          "starts": "sum"
      })
)

# Convert player values from tenths to a more readable scale
player_summary["value"] = player_summary["value"] / 10

# Efficiency metrics for comparison
player_summary["points_per_cost"] = player_summary["total_points"] / player_summary["value"]


player_summary.head()


,season,name,team,position,value,total_points,minutes,starts,points_per_cost
0,2024-25,Aaron Anselmino,Chelsea,DEF,4.000000,0,0,0,0.000000
1,2024-25,Aaron Cresswell,West Ham,DEF,3.939474,36,818,10,9.138277
2,2024-25,Aaron Hickey,Brentford,DEF,4.334211,0,0,0,0.000000
3,2024-25,Aaron Ramsdale,Arsenal,GK,4.500000,0,0,0,0.000000
4,2024-25,Aaron Ramsdale,Southampton,GK,4.400000,97,2700,30,22.045455


## Question 1: Analytical Question
Write a question about your data that can be explored with an interactive plot. A good example would be a question where you can compare different categories. Explain how the plot help answer your question and why you choose that plot type.

Examples:
- Which regions have the highest average values?
- How does one variable compare across time?

**Question:** What combination of players is expected to maximize team performance under budget constraints?

To keep the analysis simple, we use a smaller lineup instead of a full fantasy roster. Choose 5 players under a budget cap, with 1 goalkeeper, 1 defender, 2 midfielders, and 1 forward. We created the mini dashboard in quesiton 7 to answer that. The dashboard also lets the user choose a season, a scoring metric (`total_points` or `points_per_cost`), and a minimum minutes threshold so the comparison focuses on regular contributors.

This question is suited for an interactive plot because users need to compare many players at once and explore tradeoffs between cost and performance. In Question 7, we built a scatter plot places player cost on one axis and the selected performance metric on the other, while a table shows the best valid lineup under the current filters. Together, these views help the user identify efficient players and see how the recommended lineup changes as the controls change.

However, There are also some limitations. This dashboard uses a simplified 5-player lineup, and it does not include other realistic constraints such as club limits, injuries, opponent difficulty, and etc. Therefore, it is an exploratory decision-support tool rather than a full optimization model.

## Question 2. Create a simple interaction plot
Create a plot, it can be related to your question #1 or a new question, that users can interact with in some meaningful way. Explain in a markdown, how does the interaction add to the analysis.

Example of possible interactions:
*   Hover over information
*   Toogle between groups

**Question:** How does playing time relate to player output, and do these patterns look different across positions?

This simple interactive plot is showing the relationship between `minutes` played and `total_points` for players in the latest season `2025-26`. Each point represents a player, the color shows the player's position, and the point size shows player cost. Hovering reveals extra player details, and clicking legend items lets the user hide or show positions to compare groups more clearly.

In [ ]:
# Use the most recent season in the cleaned player summary table
season_q2 = sorted(player_summary["season"].dropna().unique().tolist())[-1]

# Keep players from 2025-26 season who played at least 500 minutes
scatter_q2 = player_summary[
    (player_summary["season"] == season_q2) &
    (player_summary["minutes"] >= 500)
].copy()

# Build a scatter plot of minutes played versus total points
fig_q2 = px.scatter(
    scatter_q2,
    x="minutes",
    y="total_points",
    color="position",
    size="value",
    hover_name="name",
    hover_data=["team", "starts", "value", "points_per_cost"],
    title=f"Minutes Played vs Total Points ({season_q2})",
    labels={
        "minutes": "Minutes Played",
        "total_points": "Total Points",
        "value": "Player Cost"
    }
)

fig_q2.update_layout(height=550)

fig_q2.show()

## Question 3. Choropleth Planning
Design a choropleth idea using your dataset.
In a markdown cell:
*  Identify the geographic unit you would map (state, county, country, ZIP code, etc.)
*  Identify the variable you would color by
*  Explain if any aggregation or preprocessing is needed
*  Briefly describe what GeoJSON file would be required

You do not need to have a perfect dataset for this question. However, your plan should be realistic.

If your data does not fully support a choropleth, build a prototype table that explains that structure you would need before mapping.

# Plan
  # 1. Preprocess steps
      - Look up where each player is from by checking their team name or guessing from their name. We would look at England counties.
      - Figure out the expected points from a player per game and divide that by the number of games played.
      - Make the teams based on the country they live in

  # 2. Variable Color
      - This would be the average xP divided by the players average play time per game multiplied by total game time 90 minutes.

  # 3. Whether aggregation or preprocessing is needed
      - Aggregation and preprocessing is needed because the dataset has the player's individual performance by gameweek and we want England county averages.

  # 4. GeoJSON file
    - We would need a GeoJSON file of England counties with county names that match the team locations in our dataset.

## Question 4. Geospatial Possibility Check

Determine whether your dataset can support a map-based visualization.

In a markdown cell, answer one of the following:
- If **yes**, explain what geographic fields you have and what type of map is appropriate.
- If **no**, explain what is missing and what you would need to create a map.

Write code that inspects or prepares the geographic column(s) you may use.

Our dataset cannot support a map-based visualization since this is missing geographic fields like county and team location. This has individual player performance by gameweek but no information about the players' team in specific England counties.

To create a map we would need a team_county column that maps each league team to its home area. Like "Watford" : "London".

In [ ]:
print(df_combined["team"].unique())
print("\nDataset shape:", df_combined.shape)

team_county = {
    "Arsenal": "London",
    "Aston Villa": "Birmingham",
    "Bournemouth": "Bournemouth",
    "Brighton": "Brighton",
    "Burnley": "Burn",
    "Chelsea": "London",
    "Crystal Palace": "London",
    "Everton": "Liverpool",
    "Fulham": "London",
    "Leeds": "Leeds",
    "Leicester": "Leicester",
    "Liverpool": "Liverpool",
    "Man City": "Manchester",
    "Man United": "Manchester",
    "Newcastle": "Newcastle",
    "Norwich": "Norwich",
    "Sheffield United": "Sheffield",
    'Southampton': 'Hampshire',
    'Spurs': 'London',
    'West Ham': 'London',
    'Wolves': 'West Midlands'
}

df_combined['county'] = df_combined['team'].map(team_county)

county_playerStats = df_combined.groupby('county').agg({
    'xP': 'sum',
    'minutes': 'sum'
})

county_playerStats['xP_per90min'] = (county_playerStats['xP'] / county_playerStats['minutes']) * 90

print("\nChoropleth mapping:")
print(county_playerStats.head(10))

['Bournemouth' "Nott'm Forest" 'Arsenal' 'Crystal Palace' 'Southampton'
 'Aston Villa' 'Spurs' 'Wolves' 'Brentford' 'Chelsea' 'Brighton'
 'Leicester' 'Ipswich' 'Man Utd' 'West Ham' 'Fulham' 'Liverpool'
 'Newcastle' 'Man City' 'Everton' 'Sunderland' 'Burnley' 'Leeds']

Dataset shape: (49958, 54)

Choropleth mapping:
                  xP  minutes  xP_per90min
county                                    
Birmingham    1954.3    65947     2.667096
Bournemouth   2020.0    66174     2.747303
Brighton      1739.4    65998     2.371981
Burn           303.4    28547     0.956528
Hampshire      319.3    37401     0.768348
Leeds          258.7    28619     0.813550
Leicester      759.7    37495     1.823523
Liverpool     5658.0   132021     3.857114
London       12947.2   396656     2.937679
Manchester    3004.7    66114     4.090253


## Question 5. Geopy / Location Preparation

If your dataset has location names, addresses, cities, states, or countries, use geopy on a sample of your data to geocode locations or validate location information.

If your dataset does not have data that contains location, pick 5 destination you want to visit and use geopy to geocode locations.

In [ ]:
top5_stadiums = [
    "Emirates Stadium",
    "Stamford Bridge",
    "Tottenham Hotspur Stadium",
    "Anfield",
    "Villa Park"
]

geolocation = Nominatim(user_agent="geoapi")

geo_stadiums = []

for stadium in top5_stadiums:
    location = geolocation.geocode(stadium)
    if location:
      geo_stadiums.append({'Stadium' : stadium, 'Latitude' : location.latitude, 'Longitude' : location.longitude, 'Address' : location.address})
    else:
      print(f"Location not found for {stadium}")
stadiums_df = pd.DataFrame(geo_stadiums)
print(stadiums_df)

                     Stadium   Latitude  Longitude  \
0           Emirates Stadium  51.555040  -0.108400   
1            Stamford Bridge  51.481687  -0.191034   
2  Tottenham Hotspur Stadium  51.604157  -0.066260   
3                    Anfield  53.430901  -2.960972   
4                 Villa Park  52.509126  -1.885035   

                                             Address  
0  Emirates Stadium, 75, Drayton Park, Finsbury P...  
1  Stamford Bridge, Britannia Gate, Walham Green,...  
2  Tottenham Hotspur Stadium, Paxton Terrace, Lon...  
3  Anfield, Anfield Road, Cabbage Hall, Anfield, ...  
4  Villa Park, Trinity Road, Aston, Birmingham, W...  


## Question 6. Panel Widget

Create a Panel Widget that controls something in your analysis such as the ability to choose a column, category, year, etc.

The widget should affect an output such as a plot, table, or summary statistic.

**Question:** Who are the top 10 players by total points for a selected position in a selected season?

This panel widget helps answer the question by letting the user choose both a `season` and a `position`, then updating the bar chart to show the top 10 players in that group by `total_points`. The widget makes it easy to compare categories across seasons, and the hover details provide extra context such as team, cost, minutes, and starts.

In [ ]:
# Choose a season and position, then view the top 10 players by total points
season_select_q6 = pn.widgets.Select(
    name="Season",
    options=sorted(player_summary["season"].dropna().unique().tolist()),
    value=sorted(player_summary["season"].dropna().unique().tolist())[0]
)

position_select_q6 = pn.widgets.Select(
    name="Position",
    options=["GK", "DEF", "MID", "FWD"],
    value="MID"
)

def top_players_bar_q6(season, position):
    # Filter to one season and one position, then keep the 10 highest-scoring players
    top_players = (
        player_summary[
            (player_summary["season"] == season) &
            (player_summary["position"] == position)
        ]
        .nlargest(10, "total_points")
        .sort_values("total_points", ascending=True)
    )

    fig = px.bar(
        top_players,
        x="total_points",
        y="name",
        color="team",
        orientation="h",
        hover_data=["team", "value", "minutes", "starts"],
        title=f"Top 10 {position} Players by Total Points ({season})",
        labels={"total_points": "Total Points", "name": "Player"}
    )

    fig.update_layout(height=500, yaxis={"categoryorder": "total ascending"})
    return fig

interactive_bar_q6 = pn.bind(top_players_bar_q6, season_select_q6, position_select_q6)

display(pn.Column(
    pn.Row(season_select_q6, position_select_q6),
    pn.panel(interactive_bar_q6)
))


Column
    [0] Row
        [0] Select(name='Season', options=['2024-25', '2025-26'], value='2024-25')
        [1] Select(name='Position', options=['GK', 'DEF', ...], value='MID')
    [1] ParamFunction(function, _pane=Plotly, defer_load=False)

## Question 7. Mini Dashboard

Build a small Panel dashboard with at least two components. Arrange the components so that it is in a readable layout. Your dashboard should include:
* At least one plot,
* An additional element of your choice such as a widget, table, second plot, etc.

**Question:** What combination of players is expected to maximize team performance under budget constraints?

To keep the analysis simple, we use a smaller lineup instead of a full fantasy roster. Choose 5 players under a budget cap, with 1 goalkeeper, 1 defender, 2 midfielders, and 1 forward.

In [ ]:
# Interactive controls for the mini dashboard
season_options = sorted(player_summary["season"].dropna().unique().tolist())
season_select_q7 = pn.widgets.Select(name="Season", options=season_options, value=season_options[0])
budget_slider = pn.widgets.IntSlider(name="Budget Cap", start=25, end=55, step=1, value=40)
metric_select = pn.widgets.Select(
    name="Metric",
    options={
        "Total Points": "total_points",
        "Points per Cost": "points_per_cost"
    },
    value="total_points"
)
minutes_slider = pn.widgets.IntSlider(name="Min Minutes", start=0, end=3000, step=100, value=500)

# Simplified lineup rule: 5 players total
lineup_rules = {"GK": 1, "DEF": 1, "MID": 2, "FWD": 1}

def get_filtered_players(season, min_minutes):
    # Keep only one season and only players with enough playing time
    return player_summary[
        (player_summary["season"] == season) &
        (player_summary["minutes"] >= min_minutes)
    ].copy()

def find_best_lineup(filtered, budget_cap, metric):
    # Build a small candidate pool for each position using the selected metric
    pools = {}
    for pos, required_count in lineup_rules.items():
        pos_df = filtered[filtered["position"] == pos].nlargest(8, metric)
        if len(pos_df) < required_count:
            return pd.DataFrame(), None, None
        pools[pos] = pos_df.to_dict("records")

    # Track the best valid lineup found under the budget cap
    best_lineup = None
    best_score = -np.inf
    best_cost = None

    # Try all valid 1 GK, 1 DEF, 2 MID, 1 FWD combinations
    for gk in itertools.combinations(pools["GK"], 1):
        for defender in itertools.combinations(pools["DEF"], 1):
            for mids in itertools.combinations(pools["MID"], 2):
                for fwd in itertools.combinations(pools["FWD"], 1):
                    lineup = list(gk + defender + mids + fwd)
                    total_cost = sum(player["value"] for player in lineup)
                    total_score = sum(player[metric] for player in lineup)

                    if total_cost <= budget_cap and total_score > best_score:
                        best_lineup = lineup
                        best_score = total_score
                        best_cost = total_cost

    if best_lineup is None:
        return pd.DataFrame(), None, None

    lineup_df = pd.DataFrame(best_lineup).sort_values(["position", metric], ascending=[True, False])
    return lineup_df, best_score, best_cost

def make_scatter(season, budget_cap, metric, min_minutes):
    # Create the scatter plot for the current dashboard settings
    filtered = get_filtered_players(season, min_minutes)
    metric_label = metric.replace('_', ' ').title()

    fig = px.scatter(
        filtered,
        x="value",
        y=metric,
        color="position",
        hover_data=["name", "team", "minutes", "starts", "total_points"],
        title=f"Player Cost vs {metric_label} ({season})",
        labels={"value": "Player Cost", metric: metric_label}
    )

    fig.add_vline(
        x=budget_cap / 5,
        line_dash="dash",
        line_color="gray",
        annotation_text="Average cost if budget is split evenly"
    )

    fig.update_layout(height=550)
    return fig

def lineup_table(season, budget_cap, metric, min_minutes):
    # Show the best lineup that matches the current dashboard filters
    filtered = get_filtered_players(season, min_minutes)
    lineup_df, best_score, best_cost = find_best_lineup(filtered, budget_cap, metric)

    if lineup_df.empty:
        return pn.pane.Markdown("No valid 5-player lineup was found for the current filters.")

    display_df = lineup_df[["name", "team", "position", "value", metric, "minutes", "starts"]].copy()
    summary = f"Best valid lineup score: **{best_score:.2f}**  |  Total cost: **{best_cost:.1f}**"

    return pn.Column(
        pn.pane.Markdown(summary),
        pn.pane.DataFrame(display_df, index=False, sizing_mode="stretch_width")
    )

# Connect widget values to the plotting and table functions
interactive_scatter = pn.bind(make_scatter, season_select_q7, budget_slider, metric_select, minutes_slider)
interactive_lineup = pn.bind(lineup_table, season_select_q7, budget_slider, metric_select, minutes_slider)

# Display the controls, plot, and best-lineup table together as a mini dashboard
display(pn.Column(
    pn.pane.Markdown("This mini dashboard combines a scatter plot and a lineup table. The controls let the user change the season, budget cap, scoring metric, and minimum playing time, while the two components update together."),
    pn.Row(season_select_q7, budget_slider, metric_select, minutes_slider),
    pn.panel(interactive_scatter),
    pn.pane.Markdown("### Best lineup under the current filters"),
    interactive_lineup
))

Column
    [0] Markdown(str)
    [1] Row
        [0] Select(name='Season', options=['2024-25', '2025-26'], value='2024-25')
        [1] IntSlider(end=55, name='Budget Cap', start=25, value=40)
        [2] Select(name='Metric', options={'Total Points': 'total_po...}, value='total_points')
        [3] IntSlider(end=3000, name='Min Minutes', step=100, value=500)
    [2] ParamFunction(function, _pane=Plotly, defer_load=False)
    [3] Markdown(str)
    [4] ParamFunction(function, _pane=Column, defer_load=False)

## Question 8. Reflection

Write a short reflection addressing all of the following:
- Which interactive element was most useful in your notebook?
- What was the hardest part of working with your dataset?
- Did implementing interactive tools help your analysis? Why or why not?
- If you had more time, what would you improve or add?

The most helpful one was the ability to filter by season in the graphs as this helped us compare and study consistent performers for a good price to ensure that the  FPL players can build a good team without lossing out on good players. The hardest part was preparing the dataset for visualization. For example, before we build the mini dashboard, we had to combine data from multiple seasons, keep only the useful column, and calculate the metric like point per cost. We also needed to clean and standardize fields such as team names and player positions so the filters and plots would work correctly. Implementing interactive tools did help our analysis because they allowed us to test different assumptions quickly and compare results in a more meaningful way.
If we had more time, we would improve the dashboard by adding more realistic lineup constraints and more comparison features. For example, we would include a full fantasy roster, club limits, and additional filters such as opponent difficulty, so the recommendations would be closer to real decision-making.